In [ ]:
!pip install -q datasets tokenizers tqdm xxhash pyarrow

In [ ]:
import os
import re
import json
import gc
import random
import unicodedata
import hashlib

from collections import defaultdict
from pathlib import Path

from datasets import load_dataset
from tokenizers import Tokenizer
from tqdm.auto import tqdm

In [ ]:
import os

MODEL_NAME = "best_virgo_pretrain.pt"

CHECKPOINT_PATH = None

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.strip() == MODEL_NAME:
            CHECKPOINT_PATH = os.path.join(root, file)
            break

print("Checkpoint path:", CHECKPOINT_PATH)

In [ ]:
SEED = 42
random.seed(SEED)

TOKENIZER_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-files/virgo_tokenizer.json"

OUTPUT_DIR = Path("/kaggle/working/virgo_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_TOKENS = 200_000_000

MIN_CHARS = 300
MAX_CHARS = 50_000

SHARD_TOKEN_SIZE = 5_000_000

print("Output:", OUTPUT_DIR)

In [ ]:
DOMAIN_BUDGETS = {
    "encyclopedic":  40_000_000,
    "science":       30_000_000,
    "education":     25_000_000,
    "general":       25_000_000,
    "explanation":   20_000_000,
    "technology":    20_000_000,
    "history":       10_000_000,
    "conversation":  10_000_000,
    "literature":     8_000_000,
    "stories":        6_000_000,
    "code_prose":     6_000_000,
}

assert sum(DOMAIN_BUDGETS.values()) == TARGET_TOKENS

for domain, budget in DOMAIN_BUDGETS.items():
    print(f"{domain:15s}: {budget/1e6:6.1f}M")

In [ ]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer loaded")
print("Vocab size:", tokenizer.get_vocab_size())

test = "Quantum mechanics describes physical systems."

ids = tokenizer.encode(test).ids

print(ids)
print("Tokens:", len(ids))

In [ ]:
def count_tokens(text):
    return len(tokenizer.encode(text).ids)

In [ ]:
texts = [
    "The capital of India is New Delhi.",
    "Quantum mechanics studies physical systems.",
    "Machine learning uses data to train models."
]

for text in texts:
    print(count_tokens(text), text)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return None

    text = unicodedata.normalize("NFKC", text)

    text = text.replace("\x00", " ")
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # remove emails
    text = re.sub(
        r"\b[\w\.-]+@[\w\.-]+\.\w+\b",
        " ",
        text
    )

    # collapse spaces
    text = re.sub(r"[ \t]+", " ", text)

    # maximum 2 newlines
    text = re.sub(r"\n{3,}", "\n\n", text)

    text = text.strip()

    if len(text) < MIN_CHARS:
        return None

    if len(text) > MAX_CHARS:
        text = text[:MAX_CHARS]

    return text

In [ ]:
def quality_filter(text):
    if text is None:
        return False

    length = len(text)

    if length < MIN_CHARS:
        return False

    alpha = sum(c.isalpha() for c in text)
    alpha_ratio = alpha / max(length, 1)

    if alpha_ratio < 0.55:
        return False

    words = text.split()

    if len(words) < 50:
        return False

    unique_ratio = len(set(words)) / len(words)

    if unique_ratio < 0.20:
        return False

    avg_word_length = sum(len(w) for w in words) / len(words)

    if avg_word_length < 2:
        return False

    if avg_word_length > 15:
        return False

    # excessive symbols
    symbol_count = sum(
        not c.isalnum() and
        not c.isspace() and
        c not in ".,!?;:'\"()-"
        for c in text
    )

    if symbol_count / length > 0.10:
        return False

    return True

In [ ]:
def repetition_score(text, n=3):
    words = text.lower().split()

    if len(words) < n * 2:
        return 0.0

    ngrams = [
        tuple(words[i:i+n])
        for i in range(len(words) - n + 1)
    ]

    return 1 - (
        len(set(ngrams)) /
        max(len(ngrams), 1)
    )


def repetition_filter(text):
    score = repetition_score(text)

    return score < 0.35

In [ ]:
bad = """
Robin went home Robin went home Robin went home
Robin went home Robin went home Robin went home
"""

good = """
Quantum mechanics is a physical theory that describes
matter and energy at microscopic scales.
"""

print(repetition_score(bad))
print(repetition_score(good))

In [ ]:
STORY_PATTERNS = [
    "mom smiled",
    "mom said",
    "lily said",
    "ben said",
    "they hugged",
    "the end",
    "once upon a time",
    "one day",
    "was very happy",
    "were very happy",
    "said the boy",
    "said the girl",
]

def story_bias_score(text):
    lower = text.lower()

    score = 0

    for pattern in STORY_PATTERNS:
        score += lower.count(pattern)

    return score

In [ ]:
def domain_story_filter(text, domain):
    score = story_bias_score(text)

    if domain == "stories":
        return True

    return score <= 2

In [ ]:
seen_hashes = set()

def text_hash(text):
    normalized = re.sub(
        r"\s+",
        " ",
        text.lower()
    ).strip()

    return hashlib.blake2b(
        normalized.encode("utf-8"),
        digest_size=8
    ).hexdigest()


def is_duplicate(text):
    h = text_hash(text)

    if h in seen_hashes:
        return True

    seen_hashes.add(h)

    return False

In [ ]:
stats = defaultdict(lambda: {
    "tokens": 0,
    "documents": 0,
    "rejected": 0,
    "duplicates": 0,
})


def process_text(text, domain):
    text = clean_text(text)

    if text is None:
        stats[domain]["rejected"] += 1
        return None

    if not quality_filter(text):
        stats[domain]["rejected"] += 1
        return None

    if not repetition_filter(text):
        stats[domain]["rejected"] += 1
        return None

    if not domain_story_filter(text, domain):
        stats[domain]["rejected"] += 1
        return None

    if is_duplicate(text):
        stats[domain]["duplicates"] += 1
        return None

    token_count = count_tokens(text)

    if token_count < 50:
        stats[domain]["rejected"] += 1
        return None

    return {
        "text": text,
        "domain": domain,
        "tokens": token_count,
    }

In [ ]:
class ShardWriter:
    def __init__(
        self,
        output_dir,
        domain,
        shard_token_size=5_000_000
    ):
        self.output_dir = Path(output_dir)
        self.domain = domain
        self.shard_token_size = shard_token_size

        self.domain_dir = (
            self.output_dir / domain
        )

        self.domain_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        self.shard_id = 0
        self.shard_tokens = 0
        self.file = None

        self._open_new_shard()

    def _open_new_shard(self):
        if self.file is not None:
            self.file.close()

        path = self.domain_dir / (
            f"{self.domain}_"
            f"{self.shard_id:03d}.jsonl"
        )

        self.file = open(
            path,
            "w",
            encoding="utf-8"
        )

        self.shard_tokens = 0
        self.shard_id += 1

        print("Opened:", path)

    def write(self, sample):
        if (
            self.shard_tokens +
            sample["tokens"]
            > self.shard_token_size
        ):
            self._open_new_shard()

        self.file.write(
            json.dumps(
                sample,
                ensure_ascii=False
            ) + "\n"
        )

        self.shard_tokens += sample["tokens"]

    def close(self):
        if self.file is not None:
            self.file.close()

In [ ]:
def collect_domain(
    dataset,
    domain,
    text_fn,
    token_budget,
    shuffle_buffer=10_000,
    tolerance=10_000,
):
    writer = ShardWriter(
        OUTPUT_DIR,
        domain,
        SHARD_TOKEN_SIZE
    )

    current_tokens = stats[domain]["tokens"]

    progress = tqdm(
        total=token_budget,
        initial=current_tokens,
        unit="tok",
        desc=domain
    )

    try:
        stream = dataset.shuffle(
            seed=SEED,
            buffer_size=shuffle_buffer
        )
    except Exception:
        stream = dataset

    for row in stream:

        remaining = token_budget - current_tokens

        # Stop when close enough to target
        if remaining <= tolerance:
            print(
                f"\n✅ {domain} target reached "
                f"({current_tokens:,}/{token_budget:,} tokens)"
            )
            break

        try:
            text = text_fn(row)

        except Exception:
            stats[domain]["rejected"] += 1
            continue

        sample = process_text(
            text,
            domain
        )

        if sample is None:
            continue

        remaining = token_budget - current_tokens

        if sample["tokens"] > remaining:
            continue

        writer.write(sample)

        current_tokens += sample["tokens"]

        stats[domain]["tokens"] = current_tokens
        stats[domain]["documents"] += 1

        progress.update(sample["tokens"])

    progress.close()
    writer.close()

    print()
    print(f"📚 Domain: {domain}")
    print(stats[domain])

In [ ]:
wiki = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True
)

wiki

collect_domain(
    dataset=wiki,
    domain="encyclopedic",
    text_fn=lambda x: x["text"],
    token_budget=DOMAIN_BUDGETS["encyclopedic"],
)

In [ ]:
fineweb = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

fineweb

In [ ]:
collect_domain(
    dataset=fineweb,
    domain="general",
    text_fn=lambda x: x["text"],
    token_budget=DOMAIN_BUDGETS["general"],
)

In [ ]:
fineweb_edu = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

In [ ]:
EDUCATION_KEYWORDS = [
    "learn",
    "explain",
    "concept",
    "theory",
    "example",
    "study",
    "student",
    "education",
    "understand",
    "principle",
    "method",
    "definition",
]


def educational_text(row):
    text = row["text"]

    lower = text.lower()

    score = sum(
        word in lower
        for word in EDUCATION_KEYWORDS
    )

    if score < 3:
        return ""

    return text

In [ ]:
collect_domain(
    dataset=fineweb_edu,
    domain="education",
    text_fn=educational_text,
    token_budget=DOMAIN_BUDGETS["education"],
)

In [ ]:
SCIENCE_KEYWORDS = [
    "physics",
    "quantum",
    "particle",
    "electron",
    "energy",
    "matter",
    "force",
    "mechanics",
    "thermodynamics",
    "electromagnetic",
    "biology",
    "cell",
    "protein",
    "genetic",
    "dna",
    "chemistry",
    "molecule",
    "atom",
    "reaction",
    "astronomy",
    "galaxy",
    "universe",
    "planet",
    "mathematics",
    "equation",
]


def science_text(row):
    text = row["text"]

    lower = text.lower()

    score = sum(
        keyword in lower
        for keyword in SCIENCE_KEYWORDS
    )

    if score < 3:
        return ""

    return text

In [ ]:
science_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

collect_domain(
    dataset=science_stream,
    domain="science",
    text_fn=science_text,
    token_budget=DOMAIN_BUDGETS["science"],
)

In [ ]:
TECH_KEYWORDS = [
    "computer",
    "software",
    "hardware",
    "algorithm",
    "programming",
    "python",
    "machine learning",
    "artificial intelligence",
    "neural network",
    "database",
    "processor",
    "semiconductor",
    "transistor",
    "internet",
    "network",
    "operating system",
    "compiler",
]


def technology_text(row):
    text = row["text"]

    lower = text.lower()

    score = sum(
        keyword in lower
        for keyword in TECH_KEYWORDS
    )

    if score < 2:
        return ""

    return text

In [ ]:
tech_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

collect_domain(
    dataset=tech_stream,
    domain="technology",
    text_fn=technology_text,
    token_budget=DOMAIN_BUDGETS["technology"],
)

In [ ]:
cosmo = load_dataset(
    "HuggingFaceTB/cosmopedia-100k",
    split="train",
    streaming=True
)

cosmo

In [ ]:
first = next(iter(cosmo))

print(first.keys())

for key, value in first.items():
    print("\n", key)
    print(str(value)[:300])

In [ ]:
def cosmo_text(row):
    for key in [
        "text",
        "content",
        "generation",
    ]:
        if key in row:
            if isinstance(row[key], str):
                return row[key]

    return ""

In [ ]:
cosmo_explain = load_dataset(
    "HuggingFaceTB/cosmopedia-100k",
    split="train",
    streaming=True
)

collect_domain(
    dataset=cosmo_explain,
    domain="explanation",
    text_fn=cosmo_text,
    token_budget=DOMAIN_BUDGETS["explanation"],
)

In [ ]:
HISTORY_KEYWORDS = [
    "history",
    "century",
    "empire",
    "kingdom",
    "war",
    "revolution",
    "ancient",
    "medieval",
    "civilization",
    "government",
    "political",
    "geography",
    "country",
    "population",
]


def history_text(row):
    text = row["text"]

    lower = text.lower()

    score = sum(
        keyword in lower
        for keyword in HISTORY_KEYWORDS
    )

    if score < 3:
        return ""

    return text

In [ ]:
history_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

collect_domain(
    dataset=history_stream,
    domain="history",
    text_fn=history_text,
    token_budget=DOMAIN_BUDGETS["history"],
)

In [ ]:
LITERATURE_KEYWORDS = [
    "novel",
    "poem",
    "poetry",
    "literature",
    "author",
    "character",
    "narrative",
    "prose",
    "fiction",
    "essay",
]


def literature_text(row):
    text = row["text"]

    lower = text.lower()

    score = sum(
        keyword in lower
        for keyword in LITERATURE_KEYWORDS
    )

    if score < 2:
        return ""

    return text

In [ ]:
literature_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

collect_domain(
    dataset=literature_stream,
    domain="literature",
    text_fn=literature_text,
    token_budget=DOMAIN_BUDGETS["literature"],
)

In [ ]:
STORY_KEYWORDS = [
    "once upon a time",
    "the end",
    "one day",
    "said the boy",
    "said the girl",
    "story",
]


def story_text(row):
    text = row["text"]

    lower = text.lower()

    score = sum(
        keyword in lower
        for keyword in STORY_KEYWORDS
    )

    if score < 1:
        return ""

    return text

In [ ]:
story_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

collect_domain(
    dataset=story_stream,
    domain="stories",
    text_fn=story_text,
    token_budget=DOMAIN_BUDGETS["stories"],
)

In [ ]:
CODE_PROSE_KEYWORDS = [
    "function",
    "class",
    "variable",
    "programming",
    "source code",
    "api",
    "library",
    "framework",
    "compiler",
    "runtime",
    "data structure",
    "algorithm",
    "debug",
    "implementation",
]


def code_prose_text(row):
    text = row["text"]

    lower = text.lower()

    score = sum(
        keyword in lower
        for keyword in CODE_PROSE_KEYWORDS
    )

    if score < 3:
        return ""

    return text

In [ ]:
code_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

collect_domain(
    dataset=code_stream,
    domain="code_prose",
    text_fn=code_prose_text,
    token_budget=DOMAIN_BUDGETS["code_prose"],
)

In [ ]:
def dialogue_text(row):
    text = row["text"]

    quote_count = (
        text.count('"') +
        text.count("'")
    )

    lines = text.splitlines()

    if quote_count < 6:
        return ""

    if len(lines) < 3:
        return ""

    return text

In [ ]:
conversation_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

collect_domain(
    dataset=conversation_stream,
    domain="conversation",
    text_fn=dialogue_text,
    token_budget=DOMAIN_BUDGETS["conversation"],
)

In [ ]:
import pandas as pd

stats_df = pd.DataFrame(
    [
        {
            "domain": domain,
            **values,
            "target": DOMAIN_BUDGETS[domain],
            "completion_%": (
                values["tokens"] /
                DOMAIN_BUDGETS[domain]
                * 100
            )
        }
        for domain, values in stats.items()
    ]
)

stats_df

In [ ]:
actual_tokens = sum(
    x["tokens"]
    for x in stats.values()
)

print(
    f"Total tokens: "
    f"{actual_tokens:,}"
)

print(
    f"Target: "
    f"{TARGET_TOKENS:,}"
)

print(
    f"Completion: "
    f"{actual_tokens / TARGET_TOKENS * 100:.2f}%"
)

In [ ]:
import torch
import torch.nn as nn


class ChatTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=640,
        nhead=8,
        num_layers=12,
        dim_feedforward=2048,
        seq_len=640,
        dropout=0.1,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.seq_len = seq_len

        # Token embedding
        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_embedding = nn.Embedding(
            seq_len,
            d_model
        )

        # Transformer layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        # Transformer stack
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Language model head
        self.fc_out = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        # Tie embedding and LM head weights
        self.fc_out.weight = self.embedding.weight


    def forward(self, x):

        batch_size, seq_length = x.shape

        positions = torch.arange(
            0,
            seq_length,
            device=x.device
        )

        positions = positions.unsqueeze(0)

        x = (
            self.embedding(x)
            + self.pos_embedding(positions)
        )

        # Causal attention mask
        causal_mask = torch.triu(
            torch.ones(
                seq_length,
                seq_length,
                device=x.device,
                dtype=torch.bool,
            ),
            diagonal=1,
        )

        x = self.transformer(
            x,
            mask=causal_mask,
        )

        x = self.norm(x)

        logits = self.fc_out(x)

        return logits

In [ ]:
metadata = {
    "name": "Virgo 200M Diversified Corpus",
    "target_tokens": TARGET_TOKENS,
    "tokenizer": TOKENIZER_PATH,
    "vocab_size": tokenizer.get_vocab_size(),
    "domain_budgets": DOMAIN_BUDGETS,
    "stats": dict(stats),
    "seed": SEED,
}

metadata_path = (
    OUTPUT_DIR /
    "corpus_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print(metadata_path)

In [ ]:
import glob

files = glob.glob(
    str(OUTPUT_DIR / "*" / "*.jsonl")
)

random.shuffle(files)

for file in files[:10]:

    print("\n")
    print("=" * 100)
    print(file)
    print("=" * 100)

    with open(
        file,
        "r",
        encoding="utf-8"
    ) as f:

        lines = f.readlines()

    sample = json.loads(
        random.choice(lines)
    )

    print(
        "DOMAIN:",
        sample["domain"]
    )

    print(
        "TOKENS:",
        sample["tokens"]
    )

    print()

    print(
        sample["text"][:1500]
    )

In [ ]:
CONTAMINATION_TERMS = [
    "ben said",
    "lily said",
    "mom smiled",
    "they hugged",
    "robin said",
    "the end",
]

contamination = defaultdict(int)
documents = defaultdict(int)

for file in tqdm(files):

    with open(
        file,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            sample = json.loads(line)

            domain = sample["domain"]
            text = sample["text"].lower()

            documents[domain] += 1

            if any(
                term in text
                for term in CONTAMINATION_TERMS
            ):
                contamination[domain] += 1

In [ ]:
for domain in DOMAIN_BUDGETS:

    docs = documents[domain]

    contaminated = contamination[domain]

    ratio = (
        contaminated /
        max(docs, 1)
        * 100
    )

    print(
        f"{domain:15s} "
        f"{ratio:8.4f}%"
    )

In [ ]:
manifest = []

for file in glob.glob(
    str(OUTPUT_DIR / "*" / "*.jsonl")
):

    domain = Path(file).parent.name

    manifest.append({
        "path": file,
        "domain": domain,
    })

random.shuffle(manifest)

manifest_path = (
    OUTPUT_DIR /
    "manifest.json"
)

with open(
    manifest_path,
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

print(
    "Shards:",
    len(manifest)
)

print(
    "Manifest:",
    manifest_path
)

In [ ]:
import os
import gc
import json
import glob
import random
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

In [ ]:
SEQ_LEN = 640

BATCH_SIZE = 8

GRAD_ACCUM_STEPS = 4

EPOCHS = 5

LEARNING_RATE = 5e-5

MIN_LR = 5e-6

WEIGHT_DECAY = 0.1

WARMUP_STEPS = 500

MAX_GRAD_NORM = 1.0

NUM_WORKERS = 2

SEED = 42


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("Device:", DEVICE)

In [ ]:
effective_batch = (
    BATCH_SIZE
    * GRAD_ACCUM_STEPS
)

print(
    "Effective batch size:",
    effective_batch
)

print(
    "Tokens / optimizer step:",
    effective_batch * SEQ_LEN
)

In [ ]:
CORPUS_DIR = Path(
    "/kaggle/working/virgo_200m"
)

files = glob.glob(
    str(
        CORPUS_DIR /
        "*" /
        "*.jsonl"
    )
)

print(
    "Total shards:",
    len(files)
)

for file in files[:10]:
    print(file)

In [ ]:
random.seed(SEED)

random.shuffle(files)

print(
    "First shuffled shards:"
)

for file in files[:10]:
    print(file)

In [ ]:
from pathlib import Path

import json
import numpy as np
from tqdm.auto import tqdm


CORPUS_DIR = Path(
    "/kaggle/working/virgo_v2"
)


INDEX_PATH = (
    CORPUS_DIR /
    "document_index.npy"
)


SHARD_PATHS_PATH = (
    CORPUS_DIR /
    "shard_paths.txt"
)


print("Corpus directory:", CORPUS_DIR)

In [ ]:
jsonl_files = sorted(
    CORPUS_DIR.rglob("*.jsonl")
)


print(
    "JSONL shards found:",
    len(jsonl_files)
)


if len(jsonl_files) == 0:

    raise FileNotFoundError(
        "❌ No Virgo V2 JSONL shards found"
    )


for path in jsonl_files:

    print(path)

In [ ]:
document_index = []

total_documents = 0

total_tokens = 0


for shard_id, shard_path in enumerate(
    tqdm(
        jsonl_files,
        desc="Indexing Virgo corpus"
    )
):

    with open(
        shard_path,
        "rb"
    ) as f:

        while True:

            offset = f.tell()

            line = f.readline()


            if not line:
                break


            try:

                sample = json.loads(
                    line.decode("utf-8")
                )


                if "text" not in sample:
                    continue


                if not sample["text"].strip():
                    continue


                document_index.append(
                    (
                        shard_id,
                        offset
                    )
                )


                total_documents += 1


                total_tokens += int(
                    sample.get(
                        "tokens",
                        0
                    )
                )


            except Exception:

                continue

In [ ]:
print(
    "\n🌌 VIRGO V2 INDEX STATISTICS"
)

print(
    "=" * 60
)


print(
    f"Shards     : "
    f"{len(jsonl_files):,}"
)


print(
    f"Documents  : "
    f"{total_documents:,}"
)


print(
    f"Tokens     : "
    f"{total_tokens:,}"
)


print(
    f"Index rows : "
    f"{len(document_index):,}"
)

In [ ]:
from pathlib import Path
import json
import numpy as np
from tqdm.auto import tqdm


# ============================================================
# PATHS
# ============================================================

CORPUS_DIR = Path(
    "/kaggle/working/virgo_v2"
)

INDEX_PATH = (
    CORPUS_DIR /
    "document_index.npy"
)

SHARD_PATHS_PATH = (
    CORPUS_DIR /
    "shard_paths.txt"
)


# ============================================================
# FIND ALL JSONL SHARDS
# ============================================================

jsonl_files = sorted(
    CORPUS_DIR.rglob("*.jsonl")
)


print(
    f"📚 JSONL shards found: "
    f"{len(jsonl_files)}"
)


if len(jsonl_files) == 0:

    raise FileNotFoundError(
        "❌ No JSONL shards found"
    )


# ============================================================
# RESET INDEX
# ============================================================

document_index = []

total_documents = 0
total_tokens = 0
invalid_lines = 0


# ============================================================
# BUILD DOCUMENT INDEX
# ============================================================

for shard_id, shard_path in enumerate(
    tqdm(
        jsonl_files,
        desc="🌌 Building Virgo index"
    )
):

    with open(
        shard_path,
        "rb"
    ) as f:

        while True:

            byte_offset = f.tell()

            line = f.readline()


            if not line:
                break


            try:

                sample = json.loads(
                    line
                )


                text = sample.get(
                    "text",
                    ""
                )


                if not isinstance(
                    text,
                    str
                ):

                    invalid_lines += 1
                    continue


                if not text.strip():

                    invalid_lines += 1
                    continue


                document_index.append(
                    [
                        shard_id,
                        byte_offset
                    ]
                )


                total_documents += 1


                total_tokens += int(
                    sample.get(
                        "tokens",
                        0
                    )
                )


            except Exception:

                invalid_lines += 1


# ============================================================
# CONVERT TO NUMPY
# ============================================================

document_index_np = np.asarray(
    document_index,
    dtype=np.int64
)


# ============================================================
# VERIFY
# ============================================================

print(
    "\n"
    +
    "=" * 70
)

print(
    "🌌 VIRGO V2 DOCUMENT INDEX"
)

print(
    "=" * 70
)


print(
    f"Shards        : "
    f"{len(jsonl_files):,}"
)


print(
    f"Documents     : "
    f"{total_documents:,}"
)


print(
    f"Corpus Tokens : "
    f"{total_tokens:,}"
)


print(
    f"Invalid Lines : "
    f"{invalid_lines:,}"
)


print(
    f"Index Shape   : "
    f"{document_index_np.shape}"
)


print(
    f"Index Dtype   : "
    f"{document_index_np.dtype}"
)


# ============================================================
# SAFETY CHECK
# ============================================================

if total_documents == 0:

    raise RuntimeError(
        "❌ INDEX IS EMPTY. "
        "No documents were indexed."
    )


if (
    document_index_np.ndim != 2
    or
    document_index_np.shape[1] != 2
):

    raise RuntimeError(
        f"❌ Invalid index shape: "
        f"{document_index_np.shape}"
    )


# ============================================================
# SAVE DOCUMENT INDEX
# ============================================================

np.save(
    INDEX_PATH,
    document_index_np
)


# ============================================================
# SAVE SHARD MAPPING
# ============================================================

with open(
    SHARD_PATHS_PATH,
    "w",
    encoding="utf-8"
) as f:

    for path in jsonl_files:

        f.write(
            str(path)
            +
            "\n"
        )


# ============================================================
# FINAL OUTPUT
# ============================================================

print(
    "\n✅ VIRGO INDEX CREATED"
)


print(
    "📍 Index:",
    INDEX_PATH
)


print(
    "📍 Shards:",
    SHARD_PATHS_PATH
)


print(
    "\nFirst 5 entries:"
)


print(
    document_index_np[:5]
)

In [ ]:
document_index = []

for file_id, file in enumerate(
    tqdm(
        files,
        desc="Indexing corpus"
    )
):

    with open(
        file,
        "rb"
    ) as f:

        while True:

            offset = f.tell()

            line = f.readline()

            if not line:
                break

            document_index.append(
                (
                    file_id,
                    offset
                )
            )


print(
    "Total documents:",
    len(document_index)
)

In [ ]:
from pathlib import Path
import numpy as np


CORPUS_DIR = Path(
    "/kaggle/working/virgo_v2"
)


# ==========================================
# LOAD SAVED DOCUMENT INDEX
# ==========================================

INDEX_PATH = (
    CORPUS_DIR /
    "document_index.npy"
)


document_index = np.load(
    INDEX_PATH
)


# ==========================================
# LOAD EXACT SHARD ORDER
# ==========================================

SHARD_PATHS_PATH = (
    CORPUS_DIR /
    "shard_paths.txt"
)


with open(
    SHARD_PATHS_PATH,
    "r",
    encoding="utf-8"
) as f:

    files = [
        Path(line.strip())
        for line in f
        if line.strip()
    ]


# ==========================================
# VERIFY
# ==========================================

print("🌌 Virgo V2 corpus loaded")

print(
    "Documents:",
    len(document_index)
)

print(
    "Index shape:",
    document_index.shape
)

print(
    "Shards:",
    len(files)
)

print(
    "First shard:",
    files[0]
)

print(
    "\nFirst 5 index entries:"
)

print(
    document_index[:5]
)

In [ ]:
random.seed(SEED)

random.shuffle(
    document_index
)

print(
    "Documents shuffled"
)

print(
    document_index[:10]
)

In [ ]:
from pathlib import Path
import numpy as np


# ============================================================
# VERIFY CORPUS DIRECTORY
# ============================================================

CORPUS_DIR = Path(
    "/kaggle/working/virgo_v2"
)

CORPUS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# VERIFY DOCUMENT INDEX
# ============================================================

if "document_index" not in globals():

    raise NameError(
        "❌ document_index does not exist. "
        "Run the document indexing cell first."
    )


print(
    "Documents indexed:",
    len(document_index)
)


# ============================================================
# CONVERT TO NUMPY
# ============================================================

document_index_np = np.asarray(
    document_index,
    dtype=np.int64
)


# ============================================================
# SAVE INDEX
# ============================================================

INDEX_PATH = (
    CORPUS_DIR /
    "document_index.npy"
)


np.save(
    INDEX_PATH,
    document_index_np
)


# ============================================================
# VERIFY SAVE
# ============================================================

if not INDEX_PATH.exists():

    raise FileNotFoundError(
        f"❌ Failed to save {INDEX_PATH}"
    )


print("\n✅ DOCUMENT INDEX SAVED")

print(
    "📍 Path:",
    INDEX_PATH
)

print(
    "📐 Shape:",
    document_index_np.shape
)

print(
    "🔢 Dtype:",
    document_index_np.dtype
)

print(
    "💾 Size:",
    f"{INDEX_PATH.stat().st_size / 1024**2:.2f} MB"
)

print(
    "\nFirst 5 entries:"
)

print(
    document_index_np[:5]
)

In [ ]:
from pathlib import Path
import json

CORPUS_DIR = Path("/kaggle/working/virgo_v2")

jsonl_files = sorted(
    CORPUS_DIR.rglob("*.jsonl")
)

print("JSONL files found:", len(jsonl_files))

for path in jsonl_files[:10]:
    print(path)

print("\n" + "=" * 100)

if len(jsonl_files) == 0:
    raise FileNotFoundError(
        "❌ No JSONL corpus shards found"
    )

with open(
    jsonl_files[0],
    "r",
    encoding="utf-8"
) as f:

    line = f.readline()

sample = json.loads(line)

print("FIRST FILE:")
print(jsonl_files[0])

print("\nSAMPLE KEYS:")
print(sample.keys())

print("\nSAMPLE:")
print(sample)

In [ ]:
del document_index

gc.collect()

In [ ]:
BIN_PATH = (
    CORPUS_DIR /
    "virgo_200m.bin"
)

print(BIN_PATH)

In [ ]:
document_index = np.load(
    INDEX_PATH,
    mmap_mode="r"
)

print(
    document_index.shape
)

In [ ]:
EOS_ID = tokenizer.token_to_id(
    "<eos>"
)

if EOS_ID is None:
    EOS_ID = tokenizer.token_to_id(
        "</s>"
    )


print(
    "EOS ID:",
    EOS_ID
)

In [ ]:
with open(
    BIN_PATH,
    "wb",
    buffering=1024 * 1024 * 16
) as output_file:

    total_tokens = 0

    opened_files = {}

    progress = tqdm(
        document_index,
        desc="Tokenizing corpus"
    )

    for file_id, offset in progress:

        file_id = int(file_id)

        offset = int(offset)


        if file_id not in opened_files:

            opened_files[file_id] = open(
                files[file_id],
                "rb"
            )


        f = opened_files[file_id]

        f.seek(offset)

        line = f.readline()


        try:

            sample = json.loads(
                line.decode("utf-8")
            )

        except Exception:

            continue


        text = sample["text"]


        ids = tokenizer.encode(
            text
        ).ids


        if EOS_ID is not None:

            ids.append(
                EOS_ID
            )


        token_array = np.asarray(
            ids,
            dtype=np.uint16
        )


        token_array.tofile(
            output_file
        )


        total_tokens += len(ids)


        if total_tokens % 1_000_000 < len(ids):

            progress.set_postfix(
                tokens=f"{total_tokens/1e6:.1f}M"
            )


for f in opened_files.values():
    f.close()


print(
    "Total binary tokens:",
    f"{total_tokens:,}"
)

In [ ]:
token_data = np.memmap(
    BIN_PATH,
    dtype=np.uint16,
    mode="r"
)

print(
    "Tokens:",
    f"{len(token_data):,}"
)

print(
    "Size GB:",
    token_data.nbytes / 1e9
)

print(
    "Min token:",
    token_data.min()
)

print(
    "Max token:",
    token_data.max()
)

In [ ]:
assert token_data.max() < tokenizer.get_vocab_size()

print(
    "✅ Token IDs valid"
)

In [ ]:
for _ in range(5):

    start = random.randint(
        0,
        len(token_data) - SEQ_LEN
    )

    ids = token_data[
        start:start + SEQ_LEN
    ].astype(
        np.int64
    ).tolist()


    text = tokenizer.decode(
        ids
    )


    print(
        "\n" +
        "=" * 100
    )

    print(text[:2000])

In [ ]:
class VirgoBinaryDataset(Dataset):

    def __init__(
        self,
        bin_path,
        seq_len=640
    ):

        self.seq_len = seq_len

        self.data = np.memmap(
            bin_path,
            dtype=np.uint16,
            mode="r"
        )


        self.num_sequences = (
            len(self.data) - 1
        ) // seq_len


        print(
            "Tokens:",
            f"{len(self.data):,}"
        )

        print(
            "Sequences:",
            f"{self.num_sequences:,}"
        )


    def __len__(self):

        return self.num_sequences


    def __getitem__(
        self,
        idx
    ):

        start = (
            idx *
            self.seq_len
        )


        chunk = self.data[
            start:
            start + self.seq_len + 1
        ]


        chunk = np.asarray(
            chunk,
            dtype=np.int64
        )


        x = torch.from_numpy(
            chunk[:-1].copy()
        )


        y = torch.from_numpy(
            chunk[1:].copy()
        )


        return x, y

In [ ]:
train_dataset = VirgoBinaryDataset(
    BIN_PATH,
    seq_len=SEQ_LEN
)

In [ ]:
train_size = int(
    len(train_dataset) * 0.995
)

val_size = (
    len(train_dataset)
    - train_size
)


train_data, val_data = (
    torch.utils.data.random_split(
        train_dataset,
        [
            train_size,
            val_size
        ],
        generator=torch.Generator().manual_seed(
            SEED
        )
    )
)


print(
    "Train sequences:",
    len(train_data)
)

print(
    "Validation sequences:",
    len(val_data)
)

In [ ]:
train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(
        NUM_WORKERS > 0
    ),
    drop_last=True,
)


val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(
        NUM_WORKERS > 0
    ),
    drop_last=False,
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

In [ ]:
x, y = next(
    iter(train_loader)
)

print(
    "X:",
    x.shape
)

print(
    "Y:",
    y.shape
)

print(
    "X dtype:",
    x.dtype
)


print(
    tokenizer.decode(
        x[0].tolist()
    )[:1000]
)

In [ ]:
model = ChatTransformer(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=640,
    nhead=8,
    num_layers=12,
    dim_feedforward=2048,
    seq_len=640,
    dropout=0.1,
)

total_params = sum(p.numel() for p in model.parameters())

print(f"Virgo parameters: {total_params:,}")
print(f"Virgo parameters: {total_params / 1e6:.2f}M")

In [ ]:
model = model.to(
    DEVICE
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "Parameters:",
    f"{total_params:,}"
)

print(
    "Parameters M:",
    f"{total_params/1e6:.2f}M"
)

In [ ]:
import os

MODEL_NAME = "best_virgo_pretrain.pt"

CHECKPOINT_PATH = None

for root, dirs, files in os.walk("/kaggle/input"):
    if MODEL_NAME in files:
        CHECKPOINT_PATH = os.path.join(
            root,
            MODEL_NAME
        )
        break

if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        f"❌ {MODEL_NAME} not found in /kaggle/input"
    )

print("✅ Virgo checkpoint found")
print("📍 Path:", CHECKPOINT_PATH)
print(
    "📦 Size:",
    f"{os.path.getsize(CHECKPOINT_PATH) / 1024**2:.2f} MB"
)

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

print(
    type(checkpoint)
)

In [ ]:
if isinstance(
    checkpoint,
    dict
):

    print(
        checkpoint.keys()
    )

In [ ]:
if (
    isinstance(checkpoint, dict)
    and
    "model_state_dict" in checkpoint
):

    state_dict = checkpoint[
        "model_state_dict"
    ]

else:

    state_dict = checkpoint


model.load_state_dict(
    state_dict,
    strict=True
)


print(
    "🌌 Virgo checkpoint loaded"
)

In [ ]:
import torch

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

if (
    isinstance(checkpoint, dict)
    and
    "model_state_dict" in checkpoint
):
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint


print("=" * 80)
print("🌌 VIRGO CHECKPOINT DIAGNOSTIC")
print("=" * 80)

print("\nCheckpoint type:")
print(type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    print(checkpoint.keys())


print("\nCheckpoint tensors:", len(state_dict))
print("Current model tensors:", len(model.state_dict()))


print("\n" + "=" * 80)
print("FIRST 40 CHECKPOINT LAYERS")
print("=" * 80)

for i, (name, tensor) in enumerate(
    state_dict.items()
):
    print(
        f"{i:03d} | "
        f"{name:<70} | "
        f"{tuple(tensor.shape)}"
    )

    if i >= 39:
        break


print("\n" + "=" * 80)
print("MODEL VS CHECKPOINT")
print("=" * 80)

model_state = model.state_dict()

missing = []
unexpected = []
shape_mismatch = []


for name, tensor in state_dict.items():

    if name not in model_state:

        unexpected.append(name)

    elif (
        model_state[name].shape
        !=
        tensor.shape
    ):

        shape_mismatch.append(
            (
                name,
                tuple(tensor.shape),
                tuple(model_state[name].shape)
            )
        )


for name in model_state:

    if name not in state_dict:

        missing.append(name)


print(
    "\nMissing model keys:",
    len(missing)
)

for key in missing[:20]:
    print("  ❌", key)


print(
    "\nUnexpected checkpoint keys:",
    len(unexpected)
)

for key in unexpected[:20]:
    print("  ⚠️", key)


print(
    "\nShape mismatches:",
    len(shape_mismatch)
)

for (
    name,
    checkpoint_shape,
    model_shape
) in shape_mismatch[:30]:

    print(
        f"\n{name}"
    )

    print(
        "  checkpoint:",
        checkpoint_shape
    )

    print(
        "  model     :",
        model_shape
    )

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)

In [ ]:
# ============================================================
# AUTO-DETECT VIRGO PAD TOKEN
# ============================================================

PAD_CANDIDATES = [
    "<pad>",
    "[PAD]",
    "<PAD>",
    "[pad]",
    "<|pad|>",
]


pad_token = None
pad_id = None


for candidate in PAD_CANDIDATES:

    token_id = tokenizer.token_to_id(
        candidate
    )

    if token_id is not None:

        pad_token = candidate
        pad_id = token_id

        break


if pad_id is None:

    raise ValueError(
        "❌ No PAD token found in Virgo tokenizer"
    )


print(
    "🌌 Virgo PAD token detected"
)

print(
    "PAD token:",
    pad_token
)

print(
    "PAD ID:",
    pad_id
)

In [ ]:
criterion = nn.CrossEntropyLoss(
    ignore_index=pad_id
)


print(
    "✅ Loss function ready"
)

print(
    "Ignoring:",
    pad_token,
    "ID:",
    pad_id
)

In [ ]:
steps_per_epoch = (
    len(train_loader)
    //
    GRAD_ACCUM_STEPS
)

TOTAL_STEPS = (
    steps_per_epoch
    *
    EPOCHS
)


print(
    "Optimizer steps:",
    TOTAL_STEPS
)

In [ ]:
import math


def lr_lambda(
    current_step
):

    if (
        current_step
        <
        WARMUP_STEPS
    ):

        return (
            current_step + 1
        ) / WARMUP_STEPS


    progress = (
        current_step
        -
        WARMUP_STEPS
    ) / max(
        1,
        TOTAL_STEPS
        -
        WARMUP_STEPS
    )


    cosine = (
        0.5
        *
        (
            1.0
            +
            math.cos(
                math.pi
                *
                progress
            )
        )
    )


    min_ratio = (
        MIN_LR
        /
        LEARNING_RATE
    )


    return (
        min_ratio
        +
        (
            1 - min_ratio
        )
        *
        cosine
    )


scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda
    )
)

In [ ]:
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        DEVICE == "cuda"
    )
)

In [ ]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    max_batches=100
):

    model.eval()

    total_loss = 0.0

    total_tokens = 0

    total_correct = 0


    for batch_idx, (
        x,
        y
    ) in enumerate(loader):

        if (
            batch_idx
            >=
            max_batches
        ):

            break


        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )


        with torch.amp.autocast(
            "cuda",
            enabled=(
                DEVICE == "cuda"
            )
        ):

            logits = model(x)

            loss = criterion(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                y.reshape(-1)
            )


        total_loss += loss.item()


        predictions = logits.argmax(
            dim=-1
        )


        total_correct += (
            predictions == y
        ).sum().item()


        total_tokens += y.numel()


    avg_loss = (
        total_loss
        /
        min(
            len(loader),
            max_batches
        )
    )


    accuracy = (
        total_correct
        /
        total_tokens
    )


    model.train()


    return (
        avg_loss,
        accuracy
    )

In [ ]:
# del x
# del y
# # del logits

torch.cuda.empty_cache()

model.train()

In [ ]:
CHECKPOINT_DIR = Path(
    "/kaggle/working/"
    "virgo_recovery_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def save_checkpoint(
    model,
    optimizer,
    scheduler,
    scaler,
    step,
    tokens_seen,
    loss
):

    path = (
        CHECKPOINT_DIR /
        f"virgo_"
        f"{tokens_seen//1_000_000}M.pt"
    )


    torch.save(
        {
            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "step":
                step,

            "tokens_seen":
                tokens_seen,

            "loss":
                loss,

            "config": {
                "seq_len":
                    SEQ_LEN,

                "lr":
                    LEARNING_RATE,

                "batch_size":
                    BATCH_SIZE,

                "grad_accum":
                    GRAD_ACCUM_STEPS,
            }
        },
        path
    )


    print(
        f"\n🌌 Saved: {path}"
    )

In [ ]:
import os
import math
import time
import torch

from tqdm.auto import tqdm


# ============================================================
# VIRGO V2 TRAINING CONFIG
# ============================================================

SAVE_PATH = "/kaggle/working/virgo_v2.pt"

RECOVERY_DIR = "/kaggle/working/virgo_v2_checkpoints"

os.makedirs(
    RECOVERY_DIR,
    exist_ok=True
)


CHECKPOINT_EVERY_TOKENS = 10_000_000

next_checkpoint_tokens = CHECKPOINT_EVERY_TOKENS


# ============================================================
# INITIAL TRAINING STATE
# ============================================================

global_step = 0

tokens_seen = 0

best_val_loss = float("inf")

training_start_time = time.time()


# ============================================================
# VALIDATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    val_loader
):

    model.eval()

    total_loss = 0.0

    total_correct = 0

    total_tokens = 0

    total_batches = 0


    for x, y in tqdm(
        val_loader,
        desc="Validation",
        leave=False
    ):

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )


        with torch.amp.autocast(
            "cuda",
            enabled=(DEVICE == "cuda")
        ):

            logits = model(x)

            loss = criterion(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                y.reshape(-1)
            )


        predictions = logits.argmax(
            dim=-1
        )


        valid_mask = (
            y != pad_id
        )


        correct = (
            predictions[valid_mask]
            ==
            y[valid_mask]
        ).sum().item()


        valid_tokens = (
            valid_mask.sum().item()
        )


        total_correct += correct

        total_tokens += valid_tokens

        total_loss += loss.item()

        total_batches += 1


    avg_loss = (
        total_loss
        /
        max(total_batches, 1)
    )


    accuracy = (
        total_correct
        /
        max(total_tokens, 1)
    )


    perplexity = math.exp(
        min(avg_loss, 20)
    )


    return (
        avg_loss,
        accuracy,
        perplexity
    )


# ============================================================
# CHECKPOINT SAVE FUNCTION
# ============================================================

def save_training_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    global_step,
    tokens_seen,
    val_loss=None,
    val_accuracy=None,
    val_perplexity=None
):

    torch.save(
        {
            "epoch": epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "global_step":
                global_step,

            "tokens_seen":
                tokens_seen,

            "val_loss":
                val_loss,

            "val_accuracy":
                val_accuracy,

            "val_perplexity":
                val_perplexity,

            "vocab_size":
                tokenizer.get_vocab_size(),

            "d_model":
                640,

            "seq_len":
                640,
        },
        path
    )


# ============================================================
# START TRAINING
# ============================================================

print("=" * 70)

print("🌌 VIRGO V2 CONTINUED PRETRAINING")

print("=" * 70)

print(
    f"Device          : {DEVICE}"
)

print(
    f"Epochs          : {EPOCHS}"
)

print(
    f"Gradient Accum  : {GRAD_ACCUM_STEPS}"
)

print(
    f"Checkpoint every: "
    f"{CHECKPOINT_EVERY_TOKENS / 1e6:.0f}M tokens"
)

print("=" * 70)


model.train()

optimizer.zero_grad(
    set_to_none=True
)


# ============================================================
# EPOCH LOOP
# ============================================================

for epoch in range(EPOCHS):

    epoch_start_time = time.time()

    epoch_tokens = 0

    epoch_loss_sum = 0.0

    epoch_correct = 0

    epoch_valid_tokens = 0

    epoch_batches = 0


    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        dynamic_ncols=True
    )


    # ========================================================
    # BATCH LOOP
    # ========================================================

    for batch_idx, (x, y) in enumerate(progress):


        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )


        # ====================================================
        # FORWARD PASS
        # ====================================================

        with torch.amp.autocast(
            "cuda",
            enabled=(DEVICE == "cuda")
        ):

            logits = model(x)


            raw_loss = criterion(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                y.reshape(-1)
            )


            loss = (
                raw_loss
                /
                GRAD_ACCUM_STEPS
            )


        # ====================================================
        # BACKWARD PASS
        # ====================================================

        scaler.scale(
            loss
        ).backward()


        # ====================================================
        # TOKEN ACCURACY
        # ====================================================

        with torch.no_grad():

            predictions = logits.argmax(
                dim=-1
            )


            valid_mask = (
                y != pad_id
            )


            correct_tokens = (
                predictions[valid_mask]
                ==
                y[valid_mask]
            ).sum().item()


            valid_tokens = (
                valid_mask.sum().item()
            )


        # ====================================================
        # STATISTICS
        # ====================================================

        batch_tokens = valid_tokens


        tokens_seen += batch_tokens

        epoch_tokens += batch_tokens


        epoch_correct += correct_tokens

        epoch_valid_tokens += valid_tokens


        epoch_loss_sum += raw_loss.item()

        epoch_batches += 1


        # ====================================================
        # OPTIMIZER STEP
        # ====================================================

        grad_norm = 0.0


        if (
            batch_idx + 1
        ) % GRAD_ACCUM_STEPS == 0:


            scaler.unscale_(
                optimizer
            )


            grad_norm = (
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM
                )
            )


            scaler.step(
                optimizer
            )


            scaler.update()


            optimizer.zero_grad(
                set_to_none=True
            )


            scheduler.step()


            global_step += 1


        # ====================================================
        # LIVE METRICS
        # ====================================================

        elapsed = (
            time.time()
            -
            training_start_time
        )


        tokens_per_second = (
            tokens_seen
            /
            max(elapsed, 1)
        )


        train_accuracy = (
            epoch_correct
            /
            max(epoch_valid_tokens, 1)
        )


        avg_train_loss = (
            epoch_loss_sum
            /
            max(epoch_batches, 1)
        )


        train_perplexity = math.exp(
            min(avg_train_loss, 20)
        )


        current_lr = (
            optimizer.param_groups[0]["lr"]
        )


        progress.set_postfix({

            "loss":
                f"{raw_loss.item():.4f}",

            "avg":
                f"{avg_train_loss:.4f}",

            "acc":
                f"{train_accuracy * 100:.2f}%",

            "ppl":
                f"{train_perplexity:.2f}",

            "tokens":
                f"{tokens_seen / 1e6:.1f}M",

            "tok/s":
                f"{tokens_per_second:,.0f}",

            "lr":
                f"{current_lr:.2e}",

            "grad":
                f"{float(grad_norm):.2f}",

            "step":
                global_step,

        })


        # ====================================================
        # EVERY 10M TOKENS CHECKPOINT
        # ====================================================

        if (
            tokens_seen
            >=
            next_checkpoint_tokens
        ):


            print(
                "\n"
                +
                "=" * 70
            )


            print(
                f"🌌 TOKEN CHECKPOINT "
                f"{tokens_seen / 1e6:.1f}M"
            )


            print(
                "=" * 70
            )


            val_loss, val_acc, val_ppl = evaluate(
                model,
                val_loader
            )


            print(
                f"Train Loss     : "
                f"{avg_train_loss:.4f}"
            )

            print(
                f"Train Accuracy : "
                f"{train_accuracy * 100:.2f}%"
            )

            print(
                f"Validation Loss: "
                f"{val_loss:.4f}"
            )

            print(
                f"Validation Acc : "
                f"{val_acc * 100:.2f}%"
            )

            print(
                f"Validation PPL : "
                f"{val_ppl:.2f}"
            )

            print(
                f"Tokens Seen    : "
                f"{tokens_seen:,}"
            )

            print(
                f"Global Step    : "
                f"{global_step:,}"
            )


            checkpoint_path = os.path.join(
                RECOVERY_DIR,
                f"virgo_v2_{tokens_seen // 1_000_000}M.pt"
            )


            save_training_checkpoint(
                path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch + 1,
                global_step=global_step,
                tokens_seen=tokens_seen,
                val_loss=val_loss,
                val_accuracy=val_acc,
                val_perplexity=val_ppl
            )


            print(
                f"💾 Recovery checkpoint saved"
            )

            print(
                f"📍 {checkpoint_path}"
            )


            next_checkpoint_tokens += (
                CHECKPOINT_EVERY_TOKENS
            )


            model.train()


    # ========================================================
    # EPOCH VALIDATION
    # ========================================================

    print(
        "\n"
        +
        "=" * 70
    )


    print(
        f"🌌 EPOCH {epoch + 1} COMPLETE"
    )


    print(
        "=" * 70
    )


    val_loss, val_acc, val_ppl = evaluate(
        model,
        val_loader
    )


    epoch_time = (
        time.time()
        -
        epoch_start_time
    )


    avg_train_loss = (
        epoch_loss_sum
        /
        max(epoch_batches, 1)
    )


    train_accuracy = (
        epoch_correct
        /
        max(epoch_valid_tokens, 1)
    )


    train_perplexity = math.exp(
        min(avg_train_loss, 20)
    )


    epoch_token_speed = (
        epoch_tokens
        /
        max(epoch_time, 1)
    )


    print(
        f"Train Loss       : "
        f"{avg_train_loss:.4f}"
    )

    print(
        f"Train Accuracy   : "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Train Perplexity : "
        f"{train_perplexity:.2f}"
    )


    print()


    print(
        f"Validation Loss  : "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation Acc   : "
        f"{val_acc * 100:.2f}%"
    )

    print(
        f"Validation PPL   : "
        f"{val_ppl:.2f}"
    )


    print()


    print(
        f"Epoch Tokens     : "
        f"{epoch_tokens:,}"
    )

    print(
        f"Total Tokens     : "
        f"{tokens_seen:,}"
    )

    print(
        f"Global Steps     : "
        f"{global_step:,}"
    )

    print(
        f"Token Speed      : "
        f"{epoch_token_speed:,.0f} tok/s"
    )

    print(
        f"Epoch Time       : "
        f"{epoch_time / 3600:.2f} hours"
    )


    # ========================================================
    # SAVE VIRGO V2 AFTER EVERY EPOCH
    # ========================================================

    save_training_checkpoint(
        path=SAVE_PATH,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        epoch=epoch + 1,
        global_step=global_step,
        tokens_seen=tokens_seen,
        val_loss=val_loss,
        val_accuracy=val_acc,
        val_perplexity=val_ppl
    )


    print()

    print(
        f"💾 Virgo V2 saved"
    )

    print(
        f"📍 {SAVE_PATH}"
    )


    # ========================================================
    # BEST MODEL TRACKING
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss


        best_path = (
            "/kaggle/working/"
            "best_virgo_v2.pt"
        )


        save_training_checkpoint(
            path=best_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch + 1,
            global_step=global_step,
            tokens_seen=tokens_seen,
            val_loss=val_loss,
            val_accuracy=val_acc,
            val_perplexity=val_ppl
        )


        print(
            f"🏆 New best Virgo V2"
        )

        print(
            f"Best Val Loss: "
            f"{best_val_loss:.4f}"
        )

        print(
            f"📍 {best_path}"
        )


    model.train()


# ============================================================
# TRAINING COMPLETE
# ============================================================

total_training_time = (
    time.time()
    -
    training_start_time
)


print(
    "\n"
    +
    "=" * 70
)

print(
    "🌌 VIRGO V2 CONTINUED PRETRAINING COMPLETE"
)

print(
    "=" * 70
)


print(
    f"Total Tokens : "
    f"{tokens_seen:,}"
)

print(
    f"Global Steps : "
    f"{global_step:,}"
)

print(
    f"Best Val Loss: "
    f"{best_val_loss:.4f}"
)

print(
    f"Training Time: "
    f"{total_training_time / 3600:.2f} hours"
)

print(
    f"Final Model  : "
    f"{SAVE_PATH}"
)

print("=" * 70)

In [ ]:
test_prompts = [
    "The universe is",
    "Quantum mechanics describes",
    "Artificial intelligence is",
    "The history of India",
    "A computer program is",
]


for prompt in test_prompts:

    print("\n" + "=" * 100)

    print(
        "PROMPT:",
        prompt
    )

    print("=" * 100)

    output = generate_text(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=150,
        temperature=0.75,
        top_k=40,
    )

    print(output)